In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
#import torch
import sklearn

In [2]:
dat = pd.read_csv('Data/cleaned_data.csv')
dat.columns

Index(['time', 'survey_date', 'last_4_digits_uid', 'last_name',
       'enrolled_course', 'major', 'minor', 'gender', 'gender_self_described',
       'ethnicity', 'first_gen_college', 'mother_education_level',
       'father_education_level', 'transfer_student', 'gpa_range', 'xp_courses',
       'xp_motivation', 'major_minor_motivation', 'belonging_rate',
       'hesitation_to_participate', 'respected_by_group',
       'perspective_inclusion', 'mistake_safety_in_group',
       'input_not_considered', 'comfort_asking_questions', 'meaningful_role',
       'non_valuable_contribution', 'community_work_valuable',
       'comfortable_sharing_ideas', 'feel_ignored',
       'community_partners_inclusion', 'community_partners_understanding',
       'lack_of_interaction_with_partners', 'do_patners_help', 'scenario_1',
       'scenario_1_reason', 'scenario_2', 'scenario_2_reason', 'scenario_3',
       'scenario_3_reason', 'scenario_4', 'scenario_4_reason', 'scenario_5',
       'scenario_5_reason'

In [3]:
import re

def clean_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

reason_columns = [
    'scenario_1_reason', 
    'scenario_2_reason', 
    'scenario_3_reason', 
    'scenario_4_reason', 
    'scenario_5_reason'
]

for col in reason_columns:
    dat[f'{col}_clean'] = dat[col].apply(clean_text)

print(dat[[f'{col}_clean' for col in reason_columns]].head())

                             scenario_1_reason_clean  \
0  Before acting directly on it I would go to the...   
1  I would definitely take a stand and respond to...   
2  School is very important, but I would feel a s...   
3  While it is important to acknowledge that acad...   
4  I would tell my partner to explain how they ar...   

                             scenario_2_reason_clean  \
0                I would be unsure what to say or do   
1  I would ask the instructor because they probab...   
2  I would like to get permission first to speak ...   
3  Because the community hasn't asked me to speak...   
4  if you want to speak up, you should help in an...   

                             scenario_3_reason_clean  \
0                I would take action. Ask questions.   
1  I would definitely consult the instructor. Giv...   
2  The results don’t have to necessarily be publi...   
3  Ethics, consent, and correct interpretation of...   
4  the organization is who you are serving, so

In [4]:
from transformers import pipeline
from tqdm import tqdm
tqdm.pandas()

sentiment_analyzer = pipeline(
    "sentiment-analysis", 
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,
    max_length=512
)

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [5]:
def get_continuous_sentiment(text):
    if not text or pd.isna(text) or text.strip() == "":
        return None 
    try:
        result = sentiment_analyzer(text)[0]
        label = result['label'].lower()
        score = result['score']
        
        if label == 'positive':
            return score 
        elif label == 'negative':
            return -score
        else:
            return 0.0
            
    except Exception as e:
        return None

clean_columns = [
    'scenario_1_reason_clean', 
    'scenario_2_reason_clean', 
    'scenario_3_reason_clean', 
    'scenario_4_reason_clean', 
    'scenario_5_reason_clean'
]

for col in clean_columns:
    new_col_name = col.replace('_clean', '_sentiment')
    print(f"Processing {col}...")
    dat[new_col_name] = dat[col].progress_apply(get_continuous_sentiment)

sentiment_columns = [col.replace('_clean', '_sentiment') for col in clean_columns]
dat['overall_scenario_sentiment'] = dat[sentiment_columns].mean(axis=1)

Processing scenario_1_reason_clean...


100%|██████████| 45/45 [00:05<00:00,  8.72it/s]


Processing scenario_2_reason_clean...


100%|██████████| 45/45 [00:06<00:00,  7.12it/s]


Processing scenario_3_reason_clean...


100%|██████████| 45/45 [00:04<00:00, 11.12it/s]


Processing scenario_4_reason_clean...


100%|██████████| 45/45 [00:03<00:00, 12.37it/s]


Processing scenario_5_reason_clean...


100%|██████████| 45/45 [00:09<00:00,  4.52it/s]


In [6]:
dat.head()

,time,survey_date,last_4_digits_uid,last_name,enrolled_course,major,minor,gender,gender_self_described,ethnicity,...,scenario_2_reason_clean,scenario_3_reason_clean,scenario_4_reason_clean,scenario_5_reason_clean,scenario_1_reason_sentiment,scenario_2_reason_sentiment,scenario_3_reason_sentiment,scenario_4_reason_sentiment,scenario_5_reason_sentiment,overall_scenario_sentiment
0,1/29/2026 15:02:43,1/29/2026,532,Whitney,ELTS120XP,Global studies,"Food studies, global health",Man,NaN,White,...,I would be unsure what to say or do,I would take action. Ask questions.,"I’m not sure if I read this correctly, but it ...",I think a lot of barriers come up around findi...,0.0,-0.771393,0.0,0.575386,-0.608252,-0.160852
1,1/29/2026 22:54:03,1/29/2026,7824,Sleeper,ENGCOMP130DX,Public Affairs,"Professional Writing, Environmental Systems & ...",Woman,NaN,Hispanic/Latinx,...,I would ask the instructor because they probab...,I would definitely consult the instructor. Giv...,I would advise my friend to choose Job B. This...,I would say move the workshops to zoom. This w...,0.0,0.000000,0.0,0.696884,0.569326,0.253242
2,1/30/2026 12:26:40,1/30/2026,3402,Owen,CESC 191AX,Political Science,CESC,Woman,NaN,White,...,I would like to get permission first to speak ...,The results don’t have to necessarily be publi...,I would advise my friend that they may feel mo...,In-person meetings are much more valuable ways...,0.0,0.000000,0.0,0.826918,0.765378,0.318459
3,1/30/2026 13:24:01,1/30/2026,4689,Wenn,CESC191AX,Study of Religion,Community Engagement & Social Change,Woman,NaN,White,...,Because the community hasn't asked me to speak...,"Ethics, consent, and correct interpretation of...",I think the experience of working with both in...,Zoom meetings are more convenient than in-pers...,0.0,0.000000,0.0,0.915332,0.710048,0.325076
4,1/30/2026 13:30:49,1/30/2026,2809,Sobel,CESC 191AX,Political Science,CESC,Woman,NaN,White,...,"if you want to speak up, you should help in an...","the organization is who you are serving, so if...",More opportunities to expand the scope of the ...,Offer a hybrid option where families that cann...,0.0,0.000000,0.0,0.803777,0.000000,0.160755


In [8]:
dat.to_csv('Data/dat_withSentiment.csv', index=False)

In [9]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming your dataframe is named 'df' and you have already generated the sentiment columns
sentiment_cols = [
    'scenario_1_reason_sentiment', 'scenario_2_reason_sentiment', 
    'scenario_3_reason_sentiment', 'scenario_4_reason_sentiment', 
    'scenario_5_reason_sentiment'
]

# Variables you want to keep for grouping/x-axis
metadata_cols = ['enrolled_course', 'major', 'minor', 'gender', 'gender_self_described',
       'ethnicity', 'first_gen_college', 'mother_education_level',
       'father_education_level', 'transfer_student', 'gpa_range', 'xp_courses',
       'xp_motivation', 'major_minor_motivation', 'belonging_rate',
       'hesitation_to_participate', 'respected_by_group',
       'perspective_inclusion', 'mistake_safety_in_group',
       'input_not_considered', 'comfort_asking_questions', 'meaningful_role',
       'non_valuable_contribution', 'community_work_valuable',
       'comfortable_sharing_ideas', 'feel_ignored',
       'community_partners_inclusion', 'community_partners_understanding',
       'lack_of_interaction_with_partners', 'do_patners_help', 'scenario_1',
       'scenario_1_reason', 'scenario_2', 'scenario_2_reason', 'scenario_3',
       'scenario_3_reason', 'scenario_4', 'scenario_4_reason', 'scenario_5',
       'scenario_5_reason'] 

# Reshape the data
df_long = pd.melt(
    dat, 
    id_vars=metadata_cols, 
    value_vars=sentiment_cols,
    var_name='Scenario', 
    value_name='Sentiment_Score'
)

# Clean up the 'Scenario' names for the legend (e.g., "scenario_1_sentiment" -> "Scenario 1")
df_long['Scenario'] = df_long['Scenario'].str.replace('_sentiment', '').str.replace('_', ' ').str.title()

For plotting:
- faceted plots by course, gender, etc showing sentiment score distr with the scenario that connects to the other columns
- side by side bar plots?